In [50]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import ConnectionConfig as cc
cc.setupEnvironment()

In [51]:
spark = cc.startLocalCluster("USER_DIM",4)
spark.getActiveSession()

In [52]:
from delta import configure_spark_with_delta_pip
from pyspark.sql.functions import lit, md5, concat_ws, col, to_timestamp, expr

#  initial user data
df_users = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "velo_users") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "userid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 1000) \
    .load()

#  temporary view for SQL processing
df_users.createOrReplaceTempView("operational_users")



In [53]:
# Using SQL to process the data and add SCD2 fields
df_dim_user = spark.sql("""
    SELECT
        uuid() as user_sk,
        *,
        to_timestamp('1990-01-01','yyyy-MM-dd') as scd_start,
        to_timestamp('2100-12-12','yyyy-MM-dd') as scd_end,
        md5(concat_ws('||', name, street, city, zipcode, country_code)) as md5,
        true as is_current
    FROM operational_users
""")


In [54]:
# Display the schema and sample data to confirm
df_dim_user.printSchema()
df_dim_user.show()


root
 |-- user_sk: string (nullable = false)
 |-- userid: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- street: string (nullable = true)
 |-- number: string (nullable = true)
 |-- zipcode: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- scd_start: timestamp (nullable = true)
 |-- scd_end: timestamp (nullable = true)
 |-- md5: string (nullable = false)
 |-- is_current: boolean (nullable = false)

+--------------------+------+--------------------+--------------------+--------------------+--------+-------+--------------------+------------+-------------------+-------------------+--------------------+----------+
|             user_sk|userid|                name|               email|              street|  number|zipcode|                city|country_code|          scd_start|            scd_end|                 md5|is_current|
+--------------------+------+--------------------

In [55]:
spark.sql("DROP TABLE IF EXISTS userdim")


DataFrame[]

In [56]:
# Save the transformed data as a Delta table
df_dim_user.write.format("delta").mode("overwrite").saveAsTable("userdim")


In [ ]:
#IMPLEMENTING INCREMENTAL LOGIC

In [57]:
from delta.tables import DeltaTable
from datetime import datetime

# Load the existing Delta table
from delta.tables import DeltaTable

# Load Delta table by its table name
dt_dim_user = DeltaTable.forName(spark, "userdim")


# Register as a temporary SQL view
dt_dim_user.toDF().createOrReplaceTempView("dimUser_current")




In [58]:
# Load new data from PostgreSQL
df_users_new = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "velo_users") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "userid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 1000) \
    .load()

# Register as a temporary SQL view
df_users_new.createOrReplaceTempView("operational_users_new")


In [59]:
#tranforming new data with MD5 HASH for change detection
df_users_new_transformed = spark.sql("""
    SELECT
        uuid() as source_user_sk,
        userid as source_userid,
        name as source_name,
        street as source_street,
        city as source_city,
        zipcode as source_zipcode,
        country_code as source_country_code,
        md5(concat_ws('||', name, street, city, zipcode, country_code)) as source_md5
    FROM operational_users_new
""")

# Register as a temporary SQL view
df_users_new_transformed.createOrReplaceTempView("dimUser_new")


In [60]:
# detects new or changed records by comparing hashes
detectedChanges = spark.sql("""
    SELECT *
    FROM dimUser_new source
    LEFT OUTER JOIN dimUser_current dwh
    ON dwh.userid = source.source_userid AND dwh.is_current = true
    WHERE dwh.userid IS NULL OR dwh.md5 <> source.source_md5
""")

detectedChanges.createOrReplaceTempView("detectedChanges")
detectedChanges.show()


+--------------+-------------+-----------+-------------+-----------+--------------+-------------------+----------+-------+------+----+-----+------+------+-------+----+------------+---------+-------+---+----------+
|source_user_sk|source_userid|source_name|source_street|source_city|source_zipcode|source_country_code|source_md5|user_sk|userid|name|email|street|number|zipcode|city|country_code|scd_start|scd_end|md5|is_current|
+--------------+-------------+-----------+-------------+-----------+--------------+-------------------+----------+-------+------+----+-----+------+------+-------+----+------------+---------+-------+---+----------+
+--------------+-------------+-----------+-------------+-----------+--------------+-------------------+----------+-------+------+----+-----+------+------+-------+----+------------+---------+-------+---+----------+



In [61]:

run_timestamp = datetime.now()

df_upserts = spark.sql(f"""
    SELECT
        source_user_sk AS user_sk,
        source_userid AS userid,
        source_name AS name,
        source_street AS street,
        source_city AS city,
        source_zipcode AS zipcode,
        source_country_code AS country_code,
        to_timestamp('{run_timestamp}') AS scd_start,
        to_timestamp('2100-12-12','yyyy-MM-dd') AS scd_end,
        source_md5 AS md5,
        true AS is_current
    FROM detectedChanges
    UNION
    SELECT
        user_sk,
        userid,
        name,
        street,
        city,
        zipcode,
        country_code,
        scd_start,
        to_timestamp('{run_timestamp}') AS scd_end,
        md5,
        false AS is_current
    FROM detectedChanges
    WHERE is_current IS NOT NULL
""")

df_upserts.createOrReplaceTempView("upserts")


In [62]:
spark.sql("""
    MERGE INTO dimUser_current AS target
    USING upserts AS source
    ON target.userid = source.userid AND target.is_current = true
    WHEN MATCHED THEN
        UPDATE SET scd_end = source.scd_end, is_current = source.is_current
    WHEN NOT MATCHED THEN
        INSERT (user_sk, userid, name, street, city, zipcode, country_code, scd_start, scd_end, md5, is_current)
        VALUES (source.user_sk, source.userid, source.name, source.street, source.city, source.zipcode, source.country_code, source.scd_start, source.scd_end, source.md5, source.is_current)
""")


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [63]:
# Export the updated data back to PostgreSQL
dt_dim_user.toDF().write \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "userdim") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("batchsize", 1000) \
    .mode("overwrite") \
    .save()


In [64]:
# Verify the exported data from PostgreSQL
df_verify = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "userdim") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_verify.show()



+--------------------+------+--------------------+--------------------+--------------------+--------+-------+--------------------+------------+-------------------+-------------------+--------------------+----------+
|             user_sk|userid|                name|               email|              street|  number|zipcode|                city|country_code|          scd_start|            scd_end|                 md5|is_current|
+--------------------+------+--------------------+--------------------+--------------------+--------+-------+--------------------+------------+-------------------+-------------------+--------------------+----------+
|ffd156ea-be62-48d...|     1|         Bouman Lars|Lars.Bouman@gmail...|         Somméstraat|    156 |   2060|           Antwerpen|          BE|1990-01-01 00:00:00|2100-12-12 00:00:00|d2fa521e891cbc5da...|      true|
|19420c11-1a95-41d...|     2|   van der Zee Julia|Julia.van.der.Zee...|          Europalaan|     43 |   2610| Wilrijk (Antwerpen)|      

In [65]:
spark.stop()